# 04 · Build the catalog tables

Turns the downloads of notebooks 01–03 and 03b into the tables of the competition database, saved as Parquet files in `database/` inside the project's Drive folder:

| Table | One row per |
|---|---|
| `events` | competition |
| `tasks` | task (route, start, distance, pilots launched, in ESS and in goal, day quality) |
| `turnpoints` | turnpoint of a task |
| `results` | pilot in a task's official results (glider, class, status, time, distance, score) |
| `pilots` | pilot (CIVL ID when known) |
| `tracks` | IGC file (where it is, header, fix summary, its result, and the copy it duplicates if any) |

The tables are described in `docs/database.md`.

Steps: fetch the Airtribune task results pages that are still missing, then read every task package and results page. Reading the IGC files takes about an hour; each task is cached, so **if Colab disconnects, Run All again** and it carries on.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
WORKERS = 4          # processes reading the IGC files
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive and get the code

In [ ]:
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

drive.mount("/content/drive")
auth.authenticate_user()
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
assert os.path.isdir(ROOT), f"Drive folder not found at {ROOT}"
OUT = os.path.join(ROOT, "database")
os.makedirs(OUT, exist_ok=True)
print("Downloads in", ROOT)
print("Tables go to", OUT)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)

for sub in ["flymaster_igc", "airscore_igc", "airtribune_igc"]:
    print(sub, "found" if os.path.isdir(os.path.join(ROOT, sub)) else "MISSING")


def run(*cmd):
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()


## 3. Missing Airtribune results pages

The download saved each Airtribune task's results from the `overall` category, which many events do not have. This fetches the category that lists every pilot (overall or open, otherwise the first one) for the tasks still missing. Used with the permission of the Airtribune administrators. One request every 2 seconds.

In [ ]:
run(f"{REPO}/scripts/airtribune.py", os.path.join(ROOT, "airtribune_igc"), "--results-only")

## 4. Build the tables

Prints the number of rows of each table at the end.

In [ ]:
run(f"{REPO}/scripts/build_catalog.py", ROOT, "--out", OUT, "--workers", str(WORKERS))

## 5. A first look

The tables can be queried with SQL straight from the Parquet files.

In [ ]:
import duckdb

db = duckdb.connect()
for name in ["events", "tasks", "turnpoints", "results", "pilots", "tracks"]:
    db.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{OUT}/{name}.parquet')")

db.sql("""
    SELECT t.source, substr(k.date, 1, 4) AS year, count(DISTINCT k.task_key) AS tasks, count(*) AS tracks
    FROM tracks t JOIN tasks k USING (task_key)
    WHERE t.duplicate_of IS NULL
    GROUP BY ALL ORDER BY year, t.source
""").df()

In [ ]:
db.sql("""
    SELECT coalesce(glider_cert, 'unknown') AS class, count(*) AS results
    FROM results GROUP BY ALL ORDER BY results DESC
""").df()